# kaggle-vllm 0.2.0 TP=1 / TP=2 benchmark

Status: **pending execution on a Kaggle dual-T4 notebook**. This notebook contains no saved GPU results. Attach the reviewed repository (including `scripts/benchmark_kaggle.py`), enable two T4 GPUs and Internet only if the selected model is not already cached.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = Path('/kaggle/input/kaggle-vllm-source/kaggle-vllm')
BENCHMARK = REPOSITORY / 'scripts' / 'benchmark_kaggle.py'
assert BENCHMARK.is_file(), BENCHMARK
subprocess.run(['nvidia-smi'], check=True)
subprocess.run(['nvidia-smi', 'topo', '-m'], check=True)

Run a conservative matrix first. TP=2 is not assumed to outperform TP=1; small models can be dominated by collective communication. Execute one configuration at a time so old engine workers release GPU memory before the next run.

In [ ]:
MODEL = 'facebook/opt-125m'
RESULTS = Path('/kaggle/working/kaggle-vllm-benchmarks')
RESULTS.mkdir(parents=True, exist_ok=True)

matrix = [
    (1, True, True),
    (2, True, True),
    (1, False, True),
    (2, False, True),
    (2, True, False),
]
for tp, eager, disable_car in matrix:
    name = f'tp{tp}-eager{int(eager)}-disable-car{int(disable_car)}.json'
    command = [
        sys.executable, str(BENCHMARK), '--model', MODEL,
        '--tensor-parallel-size', str(tp), '--repeats', '3',
        '--max-tokens', '128', '--output', str(RESULTS / name),
        '--enforce-eager' if eager else '--no-enforce-eager',
        '--disable-custom-all-reduce' if disable_car else '--no-disable-custom-all-reduce',
    ]
    print('RUN', name)
    subprocess.run(command, check=True)

In [ ]:
import json
for path in sorted(RESULTS.glob('*.json')):
    data = json.loads(path.read_text())
    assert data['status'] == 'executed'
    print(path.name, data['aggregate'])